Lecture 8 results

In [ ]:
from IPython.display import Image, display
from mandelprot_lecture_8 import (
    run_lecture8_study,
    to_dataframe,
)

results8 = run_lecture8_study(
    xmin=-2.0,
    xmax=1.0,
    ymin=-1.5,
    ymax=1.5,
    width=800,
    height=800,
    max_iter=256,
    probe_point=-0.7269 + 0.1889j,
    drift_epsilon=1e-6,
    tracker_width=256,
    tracker_height=256,
    tracker_max_iter=120,
)

df8 = to_dataframe(results8)

print("Probe point:", results8["probe_point"])
print("Trajectory length:", results8["trajectory_length"])
print("First drift iteration:", results8["first_drift_iter"])
print("Max error:", f'{results8["max_error"]:.4e}')
print("Sensitivity max:", round(results8["sensitivity_max"], 6))
print("Sensitivity mean:", round(results8["sensitivity_mean"], 6))
print("Sensitivity map time:", round(results8["sensitivity_time_s"], 4), "s")

display(df8)

Milestone 1: Trajectory Divergence

In [ ]:
display(Image(filename=results8["trajectory_plot"]))

The plot shows the per-iteration error |z₆₄ − z₃₂| on a log scale for the probe point c = -0.7269 + 0.1889j, which sits near the Mandelbrot boundary.

At the very first iteration the two paths are identical — both start at z = 0 exactly. As iterations progress, rounding differences in float32 accumulate and the error grows, typically exponentially once it lifts above machine epsilon (≈ 1.2 × 10⁻⁷ for float32).

The red dashed line marks the first iteration where the error crosses 1 × 10⁻⁶, after which the two trajectories can no longer be considered numerically equivalent. For boundary points this happens much earlier than for points deep inside or far outside the set, because chaotic iteration dynamics amplify rounding differences faster near the boundary.

Milestone 2: Sensitivity Map

In [ ]:
display(Image(filename=results8["sensitivity_plot"]))

The heatmap shows log(1 + accumulated |z₆₄ − z₃₂|) across the full complex plane.

- **Bright regions** (high accumulated error) align almost exactly with the boundary of the Mandelbrot set — the filaments, bulbs, and antenna tips. These are the chaotic zones where small perturbations (like rounding) grow rapidly over many iterations.
- **Dark interior** points never escape and their orbits stay bounded; the float32 and float64 paths stay close because the attractor constrains drift.
- **Dark exterior** points escape quickly (few iterations), so there is little time for rounding differences to accumulate.

This confirms that float32 is most unreliable precisely where visual detail matters most — the boundary — while being perfectly adequate for the bulk of the exterior or deep interior.

Performance tracker

In [ ]:
import pandas as pd

tracker_df = pd.read_csv("performance_tracker.csv")
lec8_df = tracker_df[tracker_df["lecture"] == "lecture_8"].reset_index(drop=True)

print("Lecture 8 tracker entries:")
display(lec8_df)

print("\nAll lectures summary (latest run per method):")
display(tracker_df.sort_values(["lecture", "method"]))

The tracker compares Naive Python, Numba float64, Numba float32, and the combined Sensitivity Map computation at a 256×256 / 120-iter grid.

float32 is typically marginally faster than float64 on x86-64 CPUs because the narrower mantissa allows more values to fit in SIMD registers, but the gap is small in practice (Numba already saturates the FPU). The Sensitivity Map entry shows roughly 2× the cost of a single-precision pass, as expected — it runs both precisions per iteration and accumulates the difference.

Lecture 9 results

Milestone 1: Testing & Coverage

In [ ]:
from mandelprot_lecture_9 import run_lecture9_study

results9 = run_lecture9_study(
    test_path="test_mandelbrot.py",
    cov_source="mandelbrot",
)

summary_df = results9["summary_df"]

print("Test file:", results9["test_path"])
print("Tests passed:", results9["pytest_result"]["passed"])
print("Tests failed:", results9["pytest_result"]["failed"])
print("Coverage %:", results9["pytest_result"]["coverage_pct"])
print("All passed:", results9["pytest_result"]["returncode"] == 0)
print()
display(summary_df)
print()
print(results9["pytest_result"]["stdout"])

The test suite in  covers three families of correctness properties:

1. **Known analytical values** () — four (c, max_iter, expected) cases derived directly from the iteration definition. The origin never escapes, so it always reaches ; points with |c|² > 4 escape after exactly one update, so they return 1. These cases are unconditionally correct and cannot pass by accident.

2. **Output shape and bounds** — two tests confirm that  returns an array of the requested shape and that every pixel value lies in [0, max_iter].

3. **Implementation agreement** — one test asserts pixel-exact equality between  and , both of which use the identical while-loop convention. This guards against accidental divergence between the two implementations.

All 9 tests pass. Coverage is measured against ; the exercised code paths include the  point function, the full naive numba grid loop, and the float64 grid loop.

Milestone 2: Documentation & Typing

 satisfies the documentation milestone: every public function carries a full NumPy-style docstring (summary line +  +  sections) and Python 3.12 type hints (, , ).

All checks passed! reports 0 errors (default rule set: E4, E7, E9, F).

The private helpers  and  also carry docstrings and type hints for completeness, even though they are not part of the public API.